In [45]:
import os
import numpy as np
import pandas as pd
import matplotlib.cm as cm
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from astropy.io import fits
import astropy.io.fits as pyfits
from astropy.wcs import WCS
from astropy.table import Table
from astropy.io import ascii
import matplotlib.pyplot as pl
import matplotlib.colors as mc
import matplotlib.patches as mpatches
from scipy.interpolate import NearestNDInterpolator
from scipy.optimize import curve_fit 
import scipy.integrate as integrate
import scipy.special as special
from scipy import stats
import scipy.stats as stats
from numpy import linspace, array, logspace, sin, cos, pi, arange, sqrt, arctan2, arccos
from mpl_toolkits.mplot3d import Axes3D
import emcee

from initial_guess_ellipse import *

from reproject import reproject_interp

import re



from collections import OrderedDict


In [47]:
tab = Table.read("/Users/danilipman/Documents/Research/UConn/CMZ_SYNTH/synth_table.tex")
absorp_tab = Table.read('/Users/danilipman/Documents/Research/UConn/CMZ_SYNTH/3d_cmz_pIII_tab2.csv')
absorp_comp_tab = ascii.read('/Users/danilipman/Documents/Research/UConn/CMZ_SYNTH/absorp_tab2.txt',format='csv')

starcount_tab = Table.read("/Users/danilipman/Documents/Research/UConn/CMZ_SYNTH/starcounts_tab.tex")

xray_tab = Table.read("/Users/danilipman/Documents/Research/UConn/CMZ_SYNTH/xray_methods.tex")
nogueras_tab = Table.read("/Users/danilipman/Documents/Research/UConn/CMZ_SYNTH/nogueras_measures.tex")

PPDF_meth_tab = Table.read("/Users/danilipman/Documents/Research/UConn/CMZ_SYNTH/paper_items/Tables/PPDF_method_info.tex")


cat_index = tab['leaf_id']
cloud_name = tab['cloud_name']
corr_coeff = tab['corr_coeff']
flux_diff = tab['flux_diff']
fdiff_stdv = tab['flux_diff_stdv']
flux_ratio = tab['flux_ratio']
fratio_stdv = tab['flux_ratio_stdv']
absorp_value = tab['absorp_value']
absorp_sigma = absorp_comp_tab['SNR_h2co']
starcount_ratio = list(starcount_tab['counts_ratio_from_avg'].copy())
starcount_ratio_stdv = list(starcount_tab['counts_ratio_stdv'].copy())
nogueras_value = nogueras_tab['LOS_dist_pc']
nogueras_sigma = nogueras_tab['sigma_pc']





In [ ]:
def corr_coeff_gauss_dist(x, mu):
    if np.isnan(mu)==True:
        return x*np.nan

    if (mu < 0.3) & (mu >= -0.3):
        sigma = 3*np.nanstd(corr_coeff)
    if (mu < -0.3) or (mu > 0.3):
        sigma = np.nanstd(corr_coeff)
    
    
    #factor = 1./( sigma * np.sqrt(2*np.pi))
    exp = np.exp(- (x - mu)**2 / (2.* (sigma**2 )))
        
    return   exp 

"""def corr_coeff_gauss_dist(x, mu,sigma):
    if (np.isnan(mu)==True):
        return x*np.nan

    elif (sigma == np.inf) | (np.isnan(sigma)==True):
        sigma = np.nanstd(3.*corr_coeff)
    
    #factor = 1./( sigma * np.sqrt(2*np.pi))
    exp = np.exp(- (x - mu)**2 / (2.* (sigma**2 )))
        
    return   exp """



def absorp_gauss_dist(x, mu, sigma):
    #sigma = 0.3*mu #sigma is 30% of the peak value
    #sigma = np.nanstd(absorp_normed, where=np.array(absorp_value)<3.5)
    #factor = 1./( sigma * np.sqrt(2*np.pi))
    exp = np.exp(- (x - mu)**2 / (2.* (sigma**2 )))
        
    return     exp 



#xray methods use step function
def hat(x,fdist, ndist):
    y=np.zeros(np.shape(x))
    
    # need to normalize so area under the curve is unity
    width = abs(fdist - ndist)
    h = 1./width
    y[(x<ndist) & (x>fdist)]+=h
    y[(x>=ndist)|(x<=fdist)]=0
    
     
    return y


#xray methods use step function
def super_gauss(x,fdist, ndist,w_scale,order):
    
    width = abs(fdist - ndist)
    h = 1/width
    center = ndist - (width/2)
    if ((x<=ndist) & (x>=fdist)): 
        return h

    if (x>ndist)|(x<fdist): 
        return h* np.exp(- ((x-center) / (width/w_scale))**(2*order) )
    
def og_super_gauss(x,fdist, ndist,w_scale,order):
    
    width = abs(fdist - ndist)
    h = 1/width
    center = ndist - (width/2)
    #if ((x<=ndist) & (x>=fdist)): 
    #    return h

    #if (x>ndist)|(x<fdist): 
    return h* np.exp(- ((x-center) / (width/w_scale))**(2*order) )

#xray methods use step function
def lognormal_tail(x,fdist, ndist):
    
    width = abs(fdist - ndist)
    h = 1/width
    centern = ndist - (width/2)
    centerf= fdist + (width/2)
    if ((x<=ndist) & (x>=fdist)): 
        return h
    if (x>ndist): 
        return h * np.exp(-(3*h/width)*(x-ndist)) #+ center+width
    if (x<fdist):
        return h * np.exp(-(3*h/width)*-(x-fdist)) #+ (center-width)
    


#For creating a posterior gaussian after the fittings
def gaussian(x, A, mu, sigma):
    return A * np.exp(- (x - mu)**2 / (2.* (sigma**2 )))



In [51]:
# convert initial ellipse y limits to (-1,1)
ring_F_y, ring_N_y =-.150,.150

x = np.arange(-100,100, 0.0001)
    
def weights(self_dict, key):
    
    
    num_keys = len(self_dict.items())-1
    w_list = []
    
    for i in self_dict.items():
        if i[0] == 'corr_coeff': 
            ww_corr = 2.
            w_list.append(ww_corr)
        if i[0] == 'absorp': 
            ww_absorp = 2.
            w_list.append(ww_absorp)
        if i[0] == 'fratio': 
            ww_fratio = 1.
            w_list.append(ww_fratio)
        if i[0] == 'fdiff': 
            ww_fdiff = 1.
            w_list.append(ww_fdiff)
        if i[0] == 'starcount': 
            ww_starcounts = 1
            w_list.append(ww_starcounts)
        if i[0] == 'nogueras': 
            ww_nogueras = 2.
            w_list.append(ww_nogueras)
        if i[0] == 'xray':  
            ww_xray = 2.
            w_list.append(ww_xray)
        
    total_w = np.sum(w_list)
    
    if key == 'corr_coeff':  return ww_corr        /total_w
    if key == 'absorp':      return ww_absorp      /total_w
    if key == 'fratio':      return ww_fratio      /total_w
    if key == 'fdiff':       return ww_fdiff       /total_w
    if key == 'starcount':   return ww_starcounts  /total_w
    if key == 'nogueras':    return ww_nogueras    /total_w
    if key == 'xray':        return ww_xray        /total_w
    if key == 'posterior':   return 1.
    
    

class ppdf:
        
    def __init__(self, cloud_id):

        x = np.arange(-100,100, 0.0001)

        cat_id = np.where(cat_index == cloud_id)[0][0]

        if np.isnan(corr_coeff[cat_id])==False:
            r_dist = corr_coeff_gauss_dist(x, corr_coeff[cat_id]) 
            r_norm = integrate.quad(corr_coeff_gauss_dist, -np.inf, np.inf, args=(corr_coeff[cat_id]))[0]
            r_dist      /=  r_norm 
            self.corr_coeff = r_dist 
            

        if np.isnan(absorp_normed[cat_id])==False:
            #absorp_dist = absorp_gauss_dist(x, absorp_normed[cat_id])
            absorp_dist = gauss_dist(x, absorp_normed[cat_id],absorp_stdv_normed[cat_id])
            absorp_norm = integrate.quad(gauss_dist, -np.inf, np.inf, args=(absorp_normed[cat_id], absorp_stdv_normed[cat_id]))[0]
            #absorp_norm = integrate.quad(absorp_gauss_dist, -np.inf, np.inf, args=(absorp_normed[cat_id]))[0]
            absorp_dist /=  absorp_norm
            self.absorp = absorp_dist 

        if np.isnan(flux_ratio_normed[cat_id])==False:
            fratio_dist = gauss_dist(x, flux_ratio_normed[cat_id],fratio_stdv_normed[cat_id])
            
            fratio_norm = integrate.quad(gauss_dist, -np.inf, np.inf, args=(flux_ratio_normed[cat_id],fratio_stdv_normed[cat_id]))[0]
            fratio_dist /=  fratio_norm 
            self.fratio = fratio_dist

        if np.isnan(flux_diff_normed[cat_id])==False:
            fdiff_dist = gauss_dist(x, flux_diff_normed[cat_id],fdiff_stdv_normed[cat_id] )   
            fdiff_norm = integrate.quad(gauss_dist, -np.inf,np.inf, args=(flux_diff_normed[cat_id],fdiff_stdv_normed[cat_id]))[0]
            fdiff_dist  /=  fdiff_norm
            self.fdiff = fdiff_dist

        if np.isnan(starcounts_normed[cat_id])==False:
            starcount_dist = gauss_dist(x, starcounts_normed[cat_id],starcounts_stdv_normed[cat_id] )    
            starcount_norm = integrate.quad(gauss_dist, -np.inf,np.inf, args=(starcounts_normed[cat_id],starcounts_stdv_normed[cat_id]))[0]
            starcount_dist  /=  starcount_norm
            self.starcount = starcount_dist
            
        #xray method for relevant clouds (modeled by tophat)
        if (cloud_id =='13') | (cloud_id =='14') :
            cloud_n = (xray_tab['xray_NEAR_los_dist_pc'][np.where(xray_tab['cloud_index']==int(cloud_id))][0])/1e3
            cloud_f = (xray_tab['xray_FAR_los_dist_pc'][np.where(xray_tab['cloud_index']==int(cloud_id))][0])/1e3
            print('xray', cloud_n, cloud_f)
            xray_hat_n, xray_hat_f = z_norming(cloud_n, ring_F_y, ring_N_y), z_norming(cloud_f, ring_F_y, ring_N_y)
            self.xray = hat(x,xray_hat_f, xray_hat_n)

        #nogueras LOS measures for 20 and 50 kms Clouds
        if (cloud_id =='9') | (cloud_id =='10') :
            nog_dist_kpc = (nogueras_value[np.where(nogueras_tab['cloud_index']==int(cloud_id))][0])/1e3
            nog_sig_kpc = (nogueras_sigma[np.where(nogueras_tab['cloud_index']==int(cloud_id))][0])/1e3
            nog_dist_normed = z_norming(nog_dist_kpc, ring_F_y, ring_N_y)
            nog_sig_normed  = stdv_z_norming(nog_sig_kpc, ring_F_y, ring_N_y,0)
            nog_dist = gauss_dist(x, nog_dist_normed,nog_sig_normed)  
            nog_norm = integrate.quad(gauss_dist, -np.inf,np.inf, args=(nog_dist_normed, nog_sig_normed))[0]
            nog_dist /= nog_norm
            self.nogueras = nog_dist


        ### Calculate the posterior PDF
        
        total_pdf = np.full(np.shape(x), None, dtype=object) 
        #total_pdf[:] = None
        
        self_dict = vars(self)
        
        uniform_pdf = np.ones(np.shape(x))
        uniform_pdf[:] = 1/np.size(x)
        
        ### Assign weights to the distributions
        ### Gaussians are best weighted by the means 
        ### other PDF shapes may be better weighted by amplitude
        for i in self_dict.items():
            #total_pdf *= i[1]
        
                 
            if i[0]=='xray'        :  
                w = weights(self_dict,i[0])
                i_weighted = i[1]*(w) / np.sum(i[1])
            else: 
                w = weights(self_dict,i[0])
                i_weighted = (i[1]*(w) + uniform_pdf*(1-w) )/ np.sum(i[1])   
       
            if total_pdf[0] == None:
                total_pdf = i_weighted
            else:
                total_pdf  *= i_weighted
            total_norm  = integrate.trapezoid(total_pdf, x)
            total_pdf  /= total_norm
            #total_pdf/=total_pdf.max()
        
        self.posterior = total_pdf


        
        
        if cloud_id == '16a': 
            self.posterior = np.empty(np.size(total_pdf))
            self.posterior[:] = np.nan 
        

CB_color_cycle = ['#377eb8', '#ff7f00', '#4daf4a','#f781bf',
                   '#a65628', '#984ea3','#999999', '#e41a1c', '#dede00']               
        

def plotting_color_label(key):
    if key == 'corr_coeff': return '#377eb8'
    if key == 'absorp': return '#e41a1c'
    if key == 'fratio': return '#4daf4a'
    if key == 'fdiff': return '#ff7f00'
    if key == 'starcount': return '#dede00'
    if key == 'xray': return '#f781bf'
    if key == 'nogueras': return '#a65628'
    if key == 'posterior': return 'black'
    #if key == 'MCMC posterior': return 'gray'

def plotting_linestyles(key):
    if key == 'corr_coeff': return '--'
    if key == 'absorp': return '-'
    if key == 'fratio': return '-.'
    if key == 'fdiff': return '--'
    if key == 'starcount': return '--'
    if key == 'xray': return '-'
    if key == 'nogueras': return '-'
    if key == 'posterior': return '-'
    #if key == 'MCMC posterior': return '-'
    



In [49]:
def gauss_dist(x, mu, sigma):
    sigma=abs(sigma)
    #factor = 1./( sigma * np.sqrt(2.*np.pi))
    exp = np.exp(- (x - mu)**2 / (2.* (sigma**2 )))
    return    exp 

##Pull out the maps, normalize the maps, get normalized stdvs to new columns
def z_norming(i, min_value, max_value):
    z_i = 2. * ((i - min_value) / (max_value - min_value)) - 1.
    return z_i

def stdv_z_norming(std_val, cnn):
    std0 = 0
    # take stdval, and 0, and normalize 
    std_val_normed =  -(z_norming(std_val, 0, 2) - cnn)
    std0_normed =  -(z_norming(std0,0, 2) - cnn)

    return abs(std_val_normed-std0_normed)

def errorprop_A_B_ratio(A,B,sigA,sigB):
    term1 = (A/B)**2
    term2 = sigA**2/A**2
    term3 = sigB**2/B**2 

    return term1 * (term2+term3)

# normalize the data value between -1 and 1 for the 
# flux difference list 
# but need to center it on -108
fdiff_cn = z_norming(-108, -150, 150)


# flux ratio is centered on 0.5
fratio_cn = z_norming(0.5, 0., 1.)

# Absorption centered on 1.
#absorp_cn = z_norming(1., np.nanmin(absorp_value), 3.5)
absorp_cn = z_norming(1., 0, 2)


# Star Count ratio centered on 1, min 0 and max 2
countratio_cn = z_norming(1., 0, 2)


flux_ratio_normed = []
flux_diff_normed = []
fdiff_stdv_normed = []
fratio_stdv_normed = []
absorp_normed = []
absorp_stdv_normed = []
starcounts_normed = []
starcounts_stdv_normed = []
counts_ratio_stdv_calc = []
corrcoeff_stderr = []

CMZ3D_dir = '/Users/danilipman/Documents/Research/UConn/3D_CMZ/Cloud_masks/'
CMZ3D_submasks_dir = '/Users/danilipman/Documents/Research/UConn/3D_CMZ/Sub_masks/'
starcount_dir = '/Users/danilipman/Documents/Research/UConn/CMZ_SYNTH/Nishiyama2013'
absorp_map_dir = '/Users/danilipman/Documents/Research/UConn/3D_CMZ/frac_abs_fits/'

def open_fits(cloudid,data):
    cat_index_dist= np.where(cat_index == cloudid)[0][0]
    cloud=cloud_name[cat_index_dist]

    
    if cloudid.isdigit() == True:

        if data == 'starcountratio':
            dir=starcount_dir
            file = pyfits.open(dir+'/%s_%s.fits'%(cloudid,data))[0]
            data = file.data
            header = WCS(file.header)
        elif data =='corr_coeff':
            dir = CMZ3D_dir
            file =  pyfits.open(dir+'%s/%s_ExtN70um_sources_to_nans_ffore0.50_cutout_smoothed_conv36_regrid_isolated.fits'%(cloud,cloud))[0]
            data =file.data
            header = WCS(file.header)
        elif data=='absorp':
            dir = absorp_map_dir
            absorp_index_dist = np.where(absorp_comp_tab['ID'] == cloudid)[0][0]
            cloud_v = absorp_comp_tab['V'][absorp_index_dist]
            idnum = int(re.findall(r'\d+', cat_index[cat_index_dist])[0])
            file = pyfits.open(dir+'%s_v%s_frac_abs.fits'%(idnum,cloud_v))[0]
            data = file.data
            header = WCS(file.header)
        else:
            dir = CMZ3D_dir
            file = pyfits.open(dir+'%s/%s_%s.fits'%(cloud,cloud,data))[0]
            data = file.data
            header = WCS(file.header)
        
        
         
    
    if cloudid.isdigit() == False:
        if data == 'starcountratio':
            dir=starcount_dir
            file = pyfits.open(dir+'/%s_%s.fits'%(cloudid,data))[0]
            data = file.data
            header = WCS(file.header)
        elif data =='corr_coeff':
            dir = CMZ3D_submasks_dir
            cloud_v = tab['v'][cat_index_dist]
            idnum = int(re.findall(r'\d+', cloudid)[0])
            file =  pyfits.open(dir+'%s/max_submask/%s_%s_ExtN70um_max_ffore0.50_conv36_regrid.fits'%(idnum,idnum,cloud_v))[0]
            data =file.data
            header = WCS(file.header)
        elif data=='absorp':
            dir = absorp_map_dir
            cloud_v = tab['v'][cat_index_dist]
            idnum = int(re.findall(r'\d+', cat_index[cat_index_dist])[0])
            file = pyfits.open(dir+'%s_v%s_frac_abs.fits'%(cloudid,cloud_v))[0]
            data = file.data
            header = WCS(file.header)
        else:
            dir = CMZ3D_submasks_dir
            cloud_v = tab['v'][cat_index_dist]
            idnum = int(re.findall(r'\d+', cloudid)[0])
            file = pyfits.open(dir+'%s/max_submask/%s_%s_%s_maxcutout_isolated.fits'%(idnum,idnum,cloud_v,data))[0]
            data = file.data
            header = WCS(file.header)



    return data,header




In [ ]:
cid = '20'

distvec = ppdf(cid)
cat_index_dist= np.where(cat_index == cid)[0][0]
cloud_v = tab['v'][cat_index_dist]
idnum = int(re.findall(r'\d+', cid)[0])

fluxdiff_data, fluxdiff_header = open_fits(cid,'flux_diff')
fluxratio_data, fluxratio_header = open_fits(cid,'flux_ratio')
#absorp_data, absorp_header = open_fits(cid,'absorp')
starcount_data, starcount_header = open_fits(cid,'starcountratio')
starcount_data[starcount_data==0] = np.nan

### normalize values before histogramming

fdiff_map_normed = z_norming(fluxdiff_data, -150, 150) - fdiff_cn
fratio_map_normed = -(z_norming(fluxratio_data, 0, 1) - fratio_cn)
starcount_map_normed = -(z_norming(starcount_data, 0, 2) - countratio_cn)
#absorp_map_normed = -(z_norming(absorp_data, 0, 2) - absorp_cn)

fdiff_stdv_n,fratio_stdv_n,starcounts_stdv_n = np.nanstd(fdiff_map_normed), stats.mad_std(fratio_map_normed[~np.isnan(fratio_map_normed)]), np.nanstd(starcount_map_normed)
